<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/Week4_LLR_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import log

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Week_4_Model_Fine_Tuning") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
df = spark.read.parquet("/content/drive/MyDrive/Parquet/02_AgriTrade_ValueOnly.parquet")
df = df.withColumn("logValue", log(df["Value"]))

In [9]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType

flagWeight = {"A":1.0, "X":1.0, "E":.7, "I":.3}
def returnWeight(x):
    return flagWeight[x]
weight_udf = udf(returnWeight, FloatType())

df = df.withColumn("Weight", weight_udf(col("Flag")))

In [10]:
year_count_df = df.groupBy("Year").count().orderBy("Year")

total_records = df.count()
target_count = total_records * 0.8
split_year = None
cumulative_count = 0

# Find the year with a split closest to 80:20
for row in year_count_df.collect():
    cumulative_count += row['count']
    diff = abs(target_count - cumulative_count)
    if cumulative_count >= target_count:
        split_year = row['Year'] if diff > abs(cumulative_count - target_count) else row['Year'] - 1
        break

# Incase no split year was found
if split_year is None:
    print("Warning: Didn't find an 80-20 split. Using last year as split_year.")
    split_year = year_count_df.agg(F.max("Year")).collect()[0][0]

train = df.filter(F.col("Year") <= split_year)
test = df.filter(F.col("Year") > split_year)

print(f"Train dataset size: {train.count()}")
print(f"Test dataset size: {test.count()}")

Train dataset size: 3421072
Test dataset size: 894329


In [11]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

country_indexer = StringIndexer(inputCol="Area", outputCol="Area_index", handleInvalid="keep")
item_indexer = StringIndexer(inputCol="Item", outputCol="Item_index", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["Area_index", "Item_index"],
    outputCols=["Area_vec", "Item_vec"]
)

assembler = VectorAssembler(
    inputCols=["Year", "Area_vec", "Item_vec"],
    outputCol="features"
)

lrModel = LinearRegression(
    featuresCol="features",
    labelCol="logValue",
    weightCol="Weight"
)
pipeline = Pipeline(stages=[
    country_indexer,
    item_indexer,
    encoder,
    assembler,
    lrModel
])

model = pipeline.fit(train)

In [12]:
lr_model = model.stages[-1]
training_summary = lr_model.summary

print(f"RMSE: {training_summary.rootMeanSquaredError}")
print(f"R2: {training_summary.r2}")

RMSE: 2.355250488165182
R2: 0.5243578743873314


In [13]:
model_path = "/content/drive/MyDrive/BigData Final Project Files/Models/Obj1_LLR_02_BaselineAltWeight"
model.write().overwrite().save(model_path)

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

lr = [stage for stage in pipeline.getStages() if isinstance(stage, LinearRegression)][0]

# ParamGrid for testing multiple hyperparams
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [10, 50, 100]) \
    .build()

evaluator = RegressionEvaluator(labelCol="logValue", predictionCol="prediction", metricName="rmse")

# CrossValidator for finding the best model
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cvModel = cv.fit(train)

In [ ]:
best_lr_model = cvModel.bestModel.stages[-1]

print("Best Linear Regression Model Parameters:")
print(f"  regParam: {best_lr_model.getRegParam()}")
print(f"  elasticNetParam: {best_lr_model.getElasticNetParam()}")
print(f"  maxIter: {best_lr_model.getMaxIter()}")

In [ ]:
cvModel = cv.fit(train)

In [16]:

# HYPERPARAMETER TUNING RESULTS TABLE

import pandas as pd

# Get the average validation RMSE for each hyperparameter combination
cv_results = []

for params, metric in zip(paramGrid, cvModel.avgMetrics):
    cv_results.append({
        "regParam": params[lr.regParam],
        "elasticNetParam": params[lr.elasticNetParam],
        "maxIter": params[lr.maxIter],
        "Validation_RMSE": metric
    })

# Convert results to pandas DataFrame
cv_results_pd = pd.DataFrame(cv_results)

# Sort by Validation RMSE
cv_results_pd = cv_results_pd.sort_values(by="Validation_RMSE")

# Display results
cv_results_pd

NameError: name 'cvModel' is not defined

In [ ]:

# VISUALIZATION: VALIDATION RMSE PER HYPERPARAMETER COMBINATION


import matplotlib.pyplot as plt

# Create readable labels for each model trial
cv_results_pd["Model_Trial"] = (
    "reg=" + cv_results_pd["regParam"].astype(str) +
    ", elastic=" + cv_results_pd["elasticNetParam"].astype(str) +
    ", iter=" + cv_results_pd["maxIter"].astype(str)
)

# Plot Validation RMSE for each hyperparameter combination
plt.figure(figsize=(12, 6))
plt.barh(cv_results_pd["Model_Trial"], cv_results_pd["Validation_RMSE"])

plt.title("Validation RMSE for LLR Hyperparameter Combinations")
plt.xlabel("Validation RMSE")
plt.ylabel("Hyperparameter Combination")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

# TEST SET EVALUATION OF BEST FINE-TUNED MODEL


from pyspark.ml.evaluation import RegressionEvaluator

# Use the best cross-validated model to predict on the test set
predictions = cvModel.transform(test)

# Create evaluators for RMSE, MAE, and R2
rmse_evaluator = RegressionEvaluator(
    labelCol="logValue",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="logValue",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="logValue",
    predictionCol="prediction",
    metricName="r2"
)

# Evaluate model performance
test_rmse = rmse_evaluator.evaluate(predictions)
test_mae = mae_evaluator.evaluate(predictions)
test_r2 = r2_evaluator.evaluate(predictions)

# Display results
print("Best Fine-Tuned Model Test Results")
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

In [ ]:

# ACTUAL VS PREDICTED TRADE TREND OVER TIME


# Group test predictions by year
# This compares the average actual and predicted log trade values per year
trend_results = predictions.groupBy("Year").agg(
    F.avg("logValue").alias("Actual_Avg_LogValue"),
    F.avg("prediction").alias("Predicted_Avg_LogValue")
).orderBy("Year")

# Convert Spark DataFrame to pandas for plotting
trend_pd = trend_results.toPandas()

# Create line graph
plt.figure(figsize=(10, 6))

plt.plot(
    trend_pd["Year"],
    trend_pd["Actual_Avg_LogValue"],
    marker="o",
    label="Actual"
)

plt.plot(
    trend_pd["Year"],
    trend_pd["Predicted_Avg_LogValue"],
    marker="o",
    label="Predicted"
)

plt.title("Actual vs Predicted Trade Trend Over Time Using Fine-Tuned LLR Model")
plt.xlabel("Year")
plt.ylabel("Average Log Trade Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
predictions = cvModel.transform(test)

# Evaluate the best model on the test set using R2
evaluator_r2 = RegressionEvaluator(labelCol="logValue", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"Test R2 for Best Model (Cross-Validated): {r2}")

In [ ]:
# SAVE THE BEST FINE-TUNED MODEL

model_path = "/content/drive/MyDrive/BigData Final Project Files/Models/Obj1_LLR_03_FineTuned"

# Save the best model selected by CrossValidator
cvModel.bestModel.write().overwrite().save(model_path)

print("Best fine-tuned model saved successfully.")